In [ ]:
# Uncomment on first run:
# !pip install snntorch -q
# !git clone https://github.com/Gorchichechka/SNN.git
import subprocess, sys
subprocess.run(["git", "clone", "https://github.com/Gorchichechka/SNN.git"],
               capture_output=True)

import numpy as np
from math import factorial
import torch
import torch.nn as nn
import torch.nn.functional as F
import snntorch as snn
from snntorch import surrogate
from SNN import ExpNeuron as en
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import warnings
warnings.filterwarnings("ignore")

Imports OK


In [ ]:
class ExpNeuronWrapper(nn.Module):
    def __init__(self, beta, threshold, membrane_zero):
        super().__init__()
        self.nrn = en.ExpNeuron(beta=beta, threshold=threshold, membrane_zero=membrane_zero,
                                learn_beta=False, learn_membrane_zero=False, learn_threshold=False)

    def forward(self, spk_input):
        mem = self.nrn.init_neuron()
        spk_out = []
        for step in range(spk_input.shape[1]):
            spk, mem = self.nrn(spk_input[:, step])
            spk_out.append(spk)
        return torch.stack(spk_out, dim=1)


class LIFNeuronWrapper(nn.Module):
    def __init__(self, beta, threshold):
        super().__init__()
        self.lif = snn.Leaky(beta=beta, threshold=threshold, spike_grad=surrogate.fast_sigmoid())

    def forward(self, spk_input):
        mem = self.lif.init_leaky()
        spk_out = []
        for step in range(spk_input.shape[1]):
            spk, mem = self.lif(spk_input[:, step], mem)
            spk_out.append(spk)
        return torch.stack(spk_out, dim=1)


def gen_spike_train(lambda_, num_steps, generator=None):
    rates = torch.full((num_steps,), float(lambda_))
    return torch.poisson(rates, generator=generator)


def psi(xi, beta, lambdas, ws):
    xi = xi.to(torch.complex128)
    r = -1j * float(beta) * xi
    for lam, w in zip(lambdas, ws):
        r = r + float(lam) * (1.0 - torch.exp(-1j * xi * w.to(torch.complex128)))
    return r


def wh_factorize(q, beta, lambdas, ws, N=4096, h=0.005):
    M = N // 2
    xi = 2 * torch.pi * torch.fft.fftfreq(N, d=h, dtype=torch.float64)
    psi_vals = psi(xi, beta, lambdas, ws)
    ln_Phi = torch.log(torch.tensor(q, dtype=torch.complex128) / (q + psi_vals))
    b = torch.fft.ifft(ln_Phi)
    idx = torch.arange(N)
    mask_plus  = ((idx >= 1) & (idx < M)).to(dtype=torch.float64)
    mask_mid   = (idx == M).to(dtype=torch.float64)
    mask_minus = (idx > M).to(dtype=torch.float64)
    c_plus  = b * (mask_plus  + mask_mid * 0.5)
    c_minus = b * (mask_minus + mask_mid * 0.5)
    phi_plus  = torch.exp(torch.fft.fft(c_plus)  - c_plus.sum())
    phi_minus = torch.exp(torch.fft.fft(c_minus) - c_minus.sum())
    return xi, phi_plus, phi_minus


def apply_Eq(f, q, beta, lambdas, ws, xi):
    symbol = q / (q + psi(xi, beta, lambdas, ws))
    return torch.fft.ifft(symbol * torch.fft.fft(f.to(torch.complex128))).real


def apply_Eq_minus(f, phi_minus):
    return torch.fft.ifft(phi_minus * torch.fft.fft(f.to(torch.complex128))).real


def pw_iterate(N_pw, T, beta, lambdas, ws, N_fft=4096, h=0.005):
    tau = T / N_pw
    q   = 1.0 / tau
    idx = torch.arange(N_fft)
    x_grid = torch.where(idx < N_fft // 2,
                         idx.to(torch.float64) * h,
                         (idx.to(torch.float64) - N_fft) * h)
    pos = x_grid > 0
    xi = 2 * torch.pi * torch.fft.fftfreq(N_fft, d=h, dtype=torch.float64)
    _, _, phi_minus = wh_factorize(q, beta, lambdas, ws, N_fft, h)
    w_iter = torch.where(pos, torch.tensor(-1.0, dtype=torch.float64),
                              torch.tensor( 0.0, dtype=torch.float64))
    for _ in range(2, N_pw + 1):
        eq = apply_Eq(w_iter, q, beta, lambdas, ws, xi)
        w_iter = torch.where(pos, eq, torch.zeros_like(eq))
    return apply_Eq_minus(w_iter, phi_minus), x_grid


def acceleration_weights(m):
    return np.array([(-1)**(m-k) * k**m / (factorial(k) * factorial(m-k))
                     for k in range(1, m+1)])


def crossing_barrier_prob(U0, threshold, T, beta, lambdas, ws,
                           N_base=5, m_acc=3, N_fft=4096, h=0.005):
    assert U0 < threshold and T > 0
    y0   = float(np.log(threshold / U0))
    i_y0 = int(round(y0 / h))
    assert 0 < i_y0 < N_fft // 2
    acc_w  = torch.tensor(acceleration_weights(m_acc), dtype=torch.float64)
    v_vals = [pw_iterate(N_base * j, T, beta, lambdas, ws, N_fft, h)[0][i_y0]
              for j in range(1, m_acc + 1)]
    return torch.clamp(1.0 + (acc_w * torch.stack(v_vals)).sum(), 0.0, 1.0)

Helpers defined


In [ ]:
SAMPLES      = 50
TEST_SAMPLES = 200
EPS          = 5e-2
BTCH_SZ      = 1
LAMBDA       = 0.55
LAMBDA_SCALE = 0.7
A, B         = 0.0, 1.0
N_FFT        = 2**14

MAX_EPOCHS   = 25
PATIENCE     = 4    # early stopping: N consecutive stagnant epochs
SEEDS        = list(range(10))

# Initial weight via softplus: softplus(W_INIT_RAW) = WEIGHT_INIT = 0.5
WEIGHT_INIT  = 0.5
W_INIT_RAW   = float(np.log(np.exp(WEIGHT_INIT) - 1 + 1e-8))  # softplus^{-1}(0.5)
print(f"W_INIT_RAW = {W_INIT_RAW:.6f}  is  softplus = {float(F.softplus(torch.tensor(W_INIT_RAW))):.6f}")


def generate_data(seed, steps):
    gnrtr = torch.Generator().manual_seed(seed)
    lambd = torch.zeros(SAMPLES).uniform_(A, B, generator=gnrtr)
    test_lambd = torch.linspace(A, B, TEST_SAMPLES)
    labels      = ((abs(lambd      - LAMBDA) <= EPS) | (lambd      > LAMBDA)).float().unsqueeze(1)
    test_labels = ((abs(test_lambd - LAMBDA) <= EPS) | (test_lambd > LAMBDA)).float().unsqueeze(1)
    trains      = torch.stack([gen_spike_train(l.item() * LAMBDA_SCALE, steps, gnrtr) for l in lambd])
    test_trains = torch.stack([gen_spike_train(l.item() * LAMBDA_SCALE, steps, gnrtr) for l in test_lambd])
    train_dlr        = DataLoader(TensorDataset(trains, labels), batch_size=BTCH_SZ)
    test_dlr         = DataLoader(TensorDataset(test_trains, test_labels), batch_size=1)
    train_lambda_dlr = DataLoader(
        TensorDataset(lambd.unsqueeze(1) * LAMBDA_SCALE, labels), batch_size=BTCH_SZ)
    return train_dlr, test_dlr, train_lambda_dlr


def compute_metrics(preds, labels):
    tp = sum(p == 1 and l == 1 for p, l in zip(preds, labels))
    tn = sum(p == 0 and l == 0 for p, l in zip(preds, labels))
    fp = sum(p == 1 and l == 0 for p, l in zip(preds, labels))
    fn = sum(p == 0 and l == 1 for p, l in zip(preds, labels))
    n  = len(preds)
    acc  = (tp + tn) / n
    prec = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec  = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


W_INIT_RAW = -0.432752  →  softplus = 0.500000
Constants & helpers defined


In [4]:
with open("best_params_exp.json") as f:
    bp_exp = json.load(f)
with open("best_params_lif.json") as f:
    bp_lif = json.load(f)

print("ExpNeuron parameters")
for k, v in bp_exp.items():
    print(f"  {k}: {v}")
print("\nLIF parameters")
for k, v in bp_lif.items():
    print(f"  {k}: {v}")

ExpNeuron parameters
  threshold: 1.0
  mem_zero: 0.2
  T: 200
  beta_exp: 0.07
  lr: 0.01

LIF parameters
  threshold: 3.0
  beta_lif: 0.85
  T: 200
  lr: 0.01


In [ ]:
def run_exp_seeds_es(bp, seeds=SEEDS):
    """
    Train ExpNeuron: fixed params, initial weight 0.5, early stopping.
    Returns: pooled_metrics, seed_metrics, all_losses, final_weights, epochs_used
    """
    all_preds, all_labels                         = [], []
    seed_metrics, all_losses, final_weights, epochs_used = [], [], [], []

    for seed in seeds:
        train_dlr, test_dlr, train_lambda_dlr = generate_data(seed, bp["T"])

        ws_raw   = nn.Parameter(torch.tensor([W_INIT_RAW], dtype=torch.float64))
        optimizer = torch.optim.Adam([ws_raw], lr=bp["lr"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

        epoch_losses = []
        best_loss    = float("inf")
        patience_ctr = 0

        for epoch in range(MAX_EPOCHS):
            ep_losses = []
            for trns, lbls in train_lambda_dlr:
                optimizer.zero_grad()
                ws = F.softplus(ws_raw)
                probs = torch.stack([
                    crossing_barrier_prob(
                        bp["mem_zero"], bp["threshold"], bp["T"], bp["beta_exp"],
                        lambdas=[lam.item()], ws=ws, N_fft=N_FFT, h=0.01
                    )
                    for lam in trns.reshape(-1)
                ])
                loss = ((probs - lbls.reshape(-1).to(torch.float64)) ** 2).mean()
                loss.backward()
                optimizer.step()
                ep_losses.append(loss.item())

            eloss = float(np.mean(ep_losses))
            epoch_losses.append(eloss)
            scheduler.step(eloss)

            if eloss < best_loss - 1e-6:
                best_loss    = eloss
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        all_losses.append(epoch_losses)
        epochs_used.append(len(epoch_losses))
        ws_val = float(F.softplus(ws_raw).detach())
        final_weights.append(ws_val)

        exp_neuron = ExpNeuronWrapper(bp["beta_exp"], bp["threshold"], bp["mem_zero"])
        s_preds, s_labels = [], []
        with torch.no_grad():
            for trn, lbl in test_dlr:
                pred = int((exp_neuron(trn * ws_val)).sum() > 0)
                s_preds.append(pred)
                s_labels.append(int(lbl.item()))
                all_preds.append(pred)
                all_labels.append(int(lbl.item()))

        seed_metrics.append(compute_metrics(s_preds, s_labels))
        print(f"  seed {seed:2d}  epochs: {len(epoch_losses):2d}  "
              f"loss: {epoch_losses[-1]:.5f}  "
              f"w: {ws_val:.4f}  "
              f"acc: {seed_metrics[-1]['accuracy']:.4f}")

    return compute_metrics(all_preds, all_labels), seed_metrics, all_losses, final_weights, epochs_used

run_exp_seeds_es defined


In [ ]:
def run_lif_seeds_es(bp, seeds=SEEDS):
    """
    Train LIF: fixed params, initial weight 0.5, early stopping.
    Returns: pooled_metrics, seed_metrics, all_losses, final_weights, epochs_used
    """
    all_preds, all_labels                         = [], []
    seed_metrics, all_losses, final_weights, epochs_used = [], [], [], []

    lif_neuron = LIFNeuronWrapper(bp["beta_lif"], bp["threshold"])

    for seed in seeds:
        train_dlr, test_dlr, _ = generate_data(seed, bp["T"])

        w_raw    = nn.Parameter(torch.tensor([W_INIT_RAW], dtype=torch.float32))
        optimizer = torch.optim.Adam([w_raw], lr=bp["lr"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)

        epoch_losses = []
        best_loss    = float("inf")
        patience_ctr = 0

        for epoch in range(MAX_EPOCHS):
            ep_losses = []
            for trns, lbls in train_dlr:
                optimizer.zero_grad()
                w       = F.softplus(w_raw)
                spk_out = lif_neuron(trns * w)
                pred    = 1 - torch.prod(1 - spk_out.float(), dim=1)
                loss    = ((pred - lbls.squeeze().float()) ** 2).mean()
                loss.backward()
                optimizer.step()
                ep_losses.append(loss.item())

            eloss = float(np.mean(ep_losses))
            epoch_losses.append(eloss)
            scheduler.step(eloss)

            if eloss < best_loss - 1e-6:
                best_loss    = eloss
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    break

        all_losses.append(epoch_losses)
        epochs_used.append(len(epoch_losses))
        w_val = float(F.softplus(w_raw).detach())
        final_weights.append(w_val)

        s_preds, s_labels = [], []
        with torch.no_grad():
            for trn, lbl in test_dlr:
                pred = int((lif_neuron(trn * w_val)).sum() > 0)
                s_preds.append(pred)
                s_labels.append(int(lbl.item()))
                all_preds.append(pred)
                all_labels.append(int(lbl.item()))

        seed_metrics.append(compute_metrics(s_preds, s_labels))
        print(f"  seed {seed:2d}  epochs: {len(epoch_losses):2d}  "
              f"loss: {epoch_losses[-1]:.5f}  "
              f"w: {w_val:.4f}  "
              f"acc: {seed_metrics[-1]['accuracy']:.4f}")

    return compute_metrics(all_preds, all_labels), seed_metrics, all_losses, final_weights, epochs_used

run_lif_seeds_es defined


In [7]:
print("=" * 62)
print("  ExpNeuron -- training (w_init=0.5, max_epochs=25, patience=4)")
print("=" * 62)
exp_pooled, exp_seed_m, exp_losses, exp_weights, exp_epochs = run_exp_seeds_es(bp_exp)

print()
print("=" * 62)
print("  LIF -- training (w_init=0.5, max_epochs=25, patience=4)")
print("=" * 62)
lif_pooled, lif_seed_m, lif_losses, lif_weights, lif_epochs = run_lif_seeds_es(bp_lif)

  ExpNeuron -- training (w_init=0.5, max_epochs=25, patience=4)
  seed  0  epochs: 25  loss: 0.02230  w: 0.2155  acc: 0.9650
  seed  1  epochs: 25  loss: 0.01902  w: 0.2164  acc: 0.9550
  seed  2  epochs: 25  loss: 0.02117  w: 0.2174  acc: 0.9450
  seed  3  epochs: 25  loss: 0.00967  w: 0.2125  acc: 0.9400
  seed  4  epochs: 25  loss: 0.00787  w: 0.2278  acc: 0.9550
  seed  5  epochs: 25  loss: 0.00779  w: 0.2155  acc: 0.9500
  seed  6  epochs: 25  loss: 0.02217  w: 0.2189  acc: 0.9250
  seed  7  epochs: 25  loss: 0.00684  w: 0.2226  acc: 0.9350
  seed  8  epochs: 10  loss: 0.01700  w: 0.2118  acc: 0.9350
  seed  9  epochs: 25  loss: 0.01998  w: 0.2130  acc: 0.9250

  LIF -- training (w_init=0.5, max_epochs=25, patience=4)
  seed  0  epochs:  5  loss: 0.06000  w: 0.5001  acc: 0.9500
  seed  1  epochs: 10  loss: 0.12000  w: 0.5551  acc: 0.9300
  seed  2  epochs:  5  loss: 0.14000  w: 0.5652  acc: 0.9400
  seed  3  epochs:  6  loss: 0.10000  w: 0.5345  acc: 0.8950
  seed  4  epochs:  5  

In [8]:
METRIC_NAMES = ["accuracy", "f1", "precision", "recall"]

print(f"\n{'':─<62}")
print(f"  FINAL METRICS  (pooled, {len(SEEDS)} seeds × {TEST_SAMPLES} test samples)")
print(f"{'':─<62}")
print(f"  {'Metric':<12}  {'ExpNeuron':^12}  {'LIF':^12}  {'Δ (Exp−LIF)':^12}")
print(f"  {'':─<55}")
for m in METRIC_NAMES:
    delta = exp_pooled[m] - lif_pooled[m]
    sign  = "+" if delta >= 0 else ""
    print(f"  {m:<12}  {exp_pooled[m]:^12.4f}  {lif_pooled[m]:^12.4f}  {sign}{delta:^11.4f}")

print(f"\n  Avg epochs:       ExpNeuron = {np.mean(exp_epochs):.1f}  |  LIF = {np.mean(lif_epochs):.1f}")
print(f"  Avg final weight: ExpNeuron = {np.mean(exp_weights):.4f}  |  LIF = {np.mean(lif_weights):.4f}")

df_seeds = pd.DataFrame({
    "seed":         SEEDS,
    "exp_accuracy": [m["accuracy"] for m in exp_seed_m],
    "exp_f1":       [m["f1"]       for m in exp_seed_m],
    "lif_accuracy": [m["accuracy"] for m in lif_seed_m],
    "lif_f1":       [m["f1"]       for m in lif_seed_m],
    "exp_epochs":   exp_epochs,
    "lif_epochs":   lif_epochs,
    "exp_weight":   [round(w, 5) for w in exp_weights],
    "lif_weight":   [round(w, 5) for w in lif_weights],
})
print(f"\n  Per-seed table:")
print(df_seeds.to_string(index=False))
df_seeds.to_csv("seed_comparison.csv", index=False)
print("\n  Saved: seed_comparison.csv")


──────────────────────────────────────────────────────────────
  FINAL METRICS  (pooled, 10 seeds × 200 test samples)
──────────────────────────────────────────────────────────────
  Metric         ExpNeuron        LIF       Δ (Exp−LIF) 
  ───────────────────────────────────────────────────────
  accuracy         0.9430        0.9145     +  0.0285   
  f1               0.9438        0.9162     +  0.0276   
  precision        0.9309        0.8982     +  0.0328   
  recall           0.9570        0.9350     +  0.0220   

  Avg epochs:       ExpNeuron = 23.5  |  LIF = 6.3
  Avg final weight: ExpNeuron = 0.2171  |  LIF = 0.5504

  Per-seed table:
 seed  exp_accuracy   exp_f1  lif_accuracy   lif_f1  exp_epochs  lif_epochs  exp_weight  lif_weight
    0         0.965 0.964824         0.950 0.948454          25           5     0.21546     0.50008
    1         0.955 0.955224         0.930 0.932692          25          10     0.21641     0.55507
    2         0.945 0.945274         0.940 0.941